In [ ]:
#Mi pregunta de investigación es: ¿Qué segmentos de consultorios están impulsando el crecimiento del "revenue" del mercado GLP-1, y cómo está cambiando la composición de ese ingreso entre sustancias a lo largo del tiempo?

In [4]:
import pandas as pd

df_final = pd.read_csv("../Práctica 1/glp1_limpio.csv")

In [ ]:
#Voy a enfocarme en las métricas de "revenue" 

columnas_revenue = ['QUANTITY', 'ITEMS', 'TOTAL_QUANTITY', 'NIC', 'ACTUAL_COST']
print(df_final[columnas_revenue].describe())

"""
NIC: media ≈ 361.89, mediana ≈ 156.96
ACTUAL_COST: media ≈ 350.93, mediana ≈ 156.97
--> Es una distribución sesgada a la derecha, lo que indica que hay algunos consultorios con ingresos significativamente más altos que la mayoría.
"""

           QUANTITY         ITEMS  TOTAL_QUANTITY           NIC   ACTUAL_COST
count  4.348743e+06  4.348743e+06    4.348743e+06  4.348743e+06  4.348743e+06
mean   7.767603e+00  3.366831e+00    2.115341e+01  3.618914e+02  3.509270e+02
std    1.317879e+01  5.606969e+00    5.665887e+01  7.308014e+02  7.223100e+02
min    0.000000e+00  1.000000e+00    0.000000e+00  0.000000e+00  0.000000e+00
25%    1.000000e+00  1.000000e+00    3.000000e+00  8.189000e+01  7.849240e+01
50%    4.000000e+00  2.000000e+00    6.000000e+00  1.569600e+02  1.569724e+02
75%    4.000000e+00  3.000000e+00    1.600000e+01  3.600000e+02  3.409494e+02
max    9.300000e+02  3.180000e+02    4.020000e+03  5.724000e+04  5.646307e+04


'\nNIC: media ≈ 361.89, mediana ≈ 156.96\nACTUAL_COST: media ≈ 350.93, mediana ≈ 156.97\n'

In [8]:
#quiero la moda de quantity y de items para ver si hay un patrón en la cantidad de productos vendidos y en el número de artículos vendidos por consultorio.

print("Moda de QUANTITY:", df_final['QUANTITY'].mode()[0])
print("Moda de ITEMS:", df_final['ITEMS'].mode()[0])

Moda de QUANTITY: 4
Moda de ITEMS: 1


In [ ]:
resumen_sustancia = df_final.groupby('SUSTANCIA')[['ACTUAL_COST', 'ITEMS']].sum()
print(resumen_sustancia.sort_values('ACTUAL_COST', ascending=False))
#Tirzepatide genera el revenue más alto (572M), pero NO tiene el mayor número de ITEMS

               ACTUAL_COST    ITEMS
SUSTANCIA                          
Tirzepatide   5.723689e+08  3401069
Semaglutide   4.332137e+08  4955543
Dulaglutide   3.812733e+08  4892294
Liraglutide   1.144619e+08  1080647
Exenatide     2.112054e+07   255509
Lixisenatide  3.652963e+06    56422


In [ ]:
#Costo promedio de cada item por sustancia
resumen_sustancia['costo_por_item'] = resumen_sustancia['ACTUAL_COST'] / resumen_sustancia['ITEMS']
print(resumen_sustancia.sort_values('costo_por_item', ascending=False))
#Tirzepatide tiene el costo promedio por item más alto , mientras que Semaglutide y dulaglutide tienen costo promedio por item más bajo. Esto sugiere que Tirzepatide está generando ingresos más altos a pesar de tener menos items vendidos, lo que podría indicar una estrategia de precios diferente o un mercado objetivo distinto.

               ACTUAL_COST    ITEMS  costo_por_item
SUSTANCIA                                          
Tirzepatide   5.723689e+08  3401069      168.290887
Liraglutide   1.144619e+08  1080647      105.919785
Semaglutide   4.332137e+08  4955543       87.420028
Exenatide     2.112054e+07   255509       82.660658
Dulaglutide   3.812733e+08  4892294       77.933447
Lixisenatide  3.652963e+06    56422       64.743595


In [ ]:
#para bajar la hipotesis planteada, necesito revisar que estemos comparando costos de la misma cantidad de quantity.
print(df_final.groupby('SUSTANCIA')['QUANTITY'].mean().sort_values(ascending=False))

#Noto que se refuta mi hipotesis y Tirzepatide no es caro "porque trae más unidades", es genuinamente más caro por unidad individual

SUSTANCIA
Semaglutide     15.111190
Dulaglutide      4.915324
Exenatide        3.399345
Liraglutide      3.228562
Lixisenatide     2.495632
Tirzepatide      1.289423
Name: QUANTITY, dtype: float64


In [ ]:
#voy a calcular el verdadero precio por unidad
resumen_sustancia_v2 = df_final.groupby('SUSTANCIA')[['ACTUAL_COST', 'TOTAL_QUANTITY']].sum()
resumen_sustancia_v2['precio_por_unidad'] = resumen_sustancia_v2['ACTUAL_COST'] / resumen_sustancia_v2['TOTAL_QUANTITY']
print(resumen_sustancia_v2.sort_values('precio_por_unidad', ascending=False))
#veo que Tirzepatide sigue siendo el más caro por unidad, mientras que Semaglutide y dulaglutide siguen siendo los más baratos. Esto refuerza la idea de que Tirzepatide está generando ingresos más altos debido a un precio por unidad más alto, en lugar de simplemente vender más unidades.

               ACTUAL_COST  TOTAL_QUANTITY  precio_por_unidad
SUSTANCIA                                                    
Tirzepatide   5.723689e+08         3730187         153.442420
Liraglutide   1.144619e+08         3129184          36.578833
Lixisenatide  3.652963e+06          134518          27.155943
Exenatide     2.112054e+07          884038          23.890989
Dulaglutide   3.812733e+08        21844529          17.453951
Semaglutide   4.332137e+08        62268305           6.957211


In [ ]:
#quiero ver cómo se estaba inflando el costo_item con el verdaddero precio por unidad, para ver cómo una sustancia se está inflando en el costo_item debido a que tiene más unidades por item.
comparacion = resumen_sustancia[['costo_por_item']].join(resumen_sustancia_v2[['precio_por_unidad']])
comparacion['diferencia'] = comparacion['costo_por_item'] - comparacion['precio_por_unidad']
print(comparacion.sort_values('diferencia', ascending=False))

              costo_por_item  precio_por_unidad  diferencia
SUSTANCIA                                                  
Semaglutide        87.420028           6.957211   80.462817
Liraglutide       105.919785          36.578833   69.340952
Dulaglutide        77.933447          17.453951   60.479496
Exenatide          82.660658          23.890989   58.769669
Lixisenatide       64.743595          27.155943   37.587652
Tirzepatide       168.290887         153.442420   14.848467


In [ ]:
"""
Tirzepatide es por mucho el más reciente de las 6 sustancias — su llegada a atención primaria
 en NHS fue apenas junio 2025, mientras que las demás (Liraglutide, Exenatide, etc.)
llevan años en el mercado y ya no tienen protección de patente tan fuerte.
 Un medicamento nuevo bajo patente, sin competencia de genéricos todavía, 
típicamente mantiene precios altos — eso podría explicar el precio
"""

In [ ]:
#Ahora haré un agrupamiento por region 

resumen_region = df_final.groupby(['REGIONAL_OFFICE_NAME'])[['ACTUAL_COST', 'ITEMS']].sum()
resumen_region['costo_por_item'] = resumen_region['ACTUAL_COST'] / resumen_region['ITEMS']
print(resumen_region.sort_values('ACTUAL_COST', ascending=False))


                           ACTUAL_COST    ITEMS  costo_por_item
REGIONAL_OFFICE_NAME                                           
MIDLANDS                  3.162964e+08  3053609      103.581160
SOUTH EAST                2.579640e+08  2322990      111.048249
NORTH EAST AND YORKSHIRE  2.309456e+08  2356473       98.004781
NORTH WEST                2.062858e+08  2106247       97.940007
LONDON                    1.906629e+08  1605531      118.753793
EAST OF ENGLAND           1.719049e+08  1721006       99.886300
SOUTH WEST                1.520317e+08  1475628      103.028507


In [ ]:
"""
London tiene el costo por item más alto, a pesar de no tener el mayor número de items vendidos. 
las tasas de prescripción son mucho más altas en zonas de mayor nivel socioeconómico,
a pesar de tener menor prevalencia de obesidad porque las zonas más ricas parecen tener 
más facilidad de acceso a estos medicamentos
(posiblemente a sustancias/presentaciones más nuevas y caras, como Tirzepatide).
"""